In [ ]:
USE ROLE ROLE_TEAM_BSJ;
USE DATABASE DB_TEAM_BSJ;

## DDL for RAW, Enhanced tables in bronze schema

In [ ]:
-- Create the 3 schemas 
CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD;

-- Verify they were created
SHOW SCHEMAS;

In [ ]:
-- =============================================
-- BRONZE LAYER 
-- =============================================

USE SCHEMA BRONZE;

-- 1. RAW TABLE - Mirror of CSV structure
CREATE OR REPLACE TABLE disney_plus_shows_raw (
    imdb_id VARCHAR PRIMARY KEY,
    title VARCHAR,
    plot VARCHAR,
    type VARCHAR,
    rated VARCHAR,
    year VARCHAR,
    released_at VARCHAR,
    added_at VARCHAR,
    runtime VARCHAR,
    genre VARCHAR,
    director VARCHAR,
    writer VARCHAR,
    actors VARCHAR,
    language VARCHAR,
    country VARCHAR,
    awards VARCHAR,
    metascore VARCHAR,
    imdb_rating VARCHAR,
    imdb_votes VARCHAR
);



In [ ]:
USE SCHEMA BRONZE;
-- 2. ENHANCED TABLE - All transformations in one table
CREATE OR REPLACE TABLE disney_plus_shows_enhanced (
    imdb_id VARCHAR PRIMARY KEY,
    title VARCHAR,
    plot VARCHAR,
    type VARCHAR,
    rated VARCHAR,
    
    -- Cleaned temporal data
    year INTEGER,
    release_date DATE,
    added_date DATE,
    runtime_minutes INTEGER,
    
    -- Original multi-value columns (preserved)
    genre VARCHAR,
    director VARCHAR,
    writer VARCHAR,
    actors VARCHAR,
    language VARCHAR,
    country VARCHAR,
    awards VARCHAR,
    
    -- Awards parsing
    total_wins INTEGER,
    total_nominations INTEGER,
    is_oscar_winner BOOLEAN,
    is_oscar_nominee BOOLEAN,
    is_emmy_winner BOOLEAN,
    is_emmy_nominee BOOLEAN,
    is_goldenglobe_winner BOOLEAN,
    is_goldenglobe_nominee BOOLEAN,
    
    -- Cleaned numerical metrics
    metascore INTEGER,
    imdb_rating NUMBER(3,1),
    imdb_votes INTEGER,
    
    -- AI-generated content
    plot_sentiment FLOAT,
    plot_summary VARCHAR,
    sentiment_category VARCHAR
);

## Stream on enhanced table to keep a track of incremental data

In [ ]:
-- Create the stream on your source table
CREATE OR REPLACE STREAM BRONZE.disney_plus_shows_enhanced_stream
ON TABLE BRONZE.disney_plus_shows_enhanced
APPEND_ONLY = TRUE;

## DDL for DIM, FACT, COMPOSITE tables in SILVER layer

In [ ]:
--==============================================================
-- 0. SET CONTEXT
--==============================================================
USE SCHEMA SILVER;

--==============================================================
-- 1. SEQUENCE GENERATOR
--==============================================================
CREATE OR REPLACE SEQUENCE SILVER.seq_show_key
  START = 1
  INCREMENT = 1
  COMMENT = 'Sequence for generating primary keys for DIM_SHOWS';

--==============================================================
-- 2. LOOKUP DIMENSIONS (Has-Many)
--==============================================================
CREATE OR REPLACE TABLE SILVER.DIM_GENRES (
    genre_key NUMBER AUTOINCREMENT PRIMARY KEY,
    genre_name VARCHAR UNIQUE NOT NULL
);

CREATE OR REPLACE TABLE SILVER.DIM_PERSONS (
    person_key NUMBER AUTOINCREMENT PRIMARY KEY,
    person_name VARCHAR UNIQUE NOT NULL
);

CREATE OR REPLACE TABLE SILVER.DIM_LANGUAGES (
    language_key NUMBER AUTOINCREMENT PRIMARY KEY,
    language_name VARCHAR UNIQUE NOT NULL
);

CREATE OR REPLACE TABLE SILVER.DIM_COUNTRIES (
    country_key NUMBER AUTOINCREMENT PRIMARY KEY,
    country_name VARCHAR UNIQUE NOT NULL
);

--==============================================================
-- 3. LOOKUP DIMENSIONS (Has-One)
--==============================================================
CREATE OR REPLACE TABLE SILVER.DIM_TYPES (
    type_key NUMBER AUTOINCREMENT PRIMARY KEY,
    type_name VARCHAR UNIQUE NOT NULL
);

CREATE OR REPLACE TABLE SILVER.DIM_RATINGS (
    rating_key NUMBER AUTOINCREMENT PRIMARY KEY,
    rating_name VARCHAR UNIQUE NOT NULL
);

--==============================================================
-- 4. CORE DIMENSION (DIM_SHOWS)
-- Now includes all date columns directly!
--==============================================================
CREATE OR REPLACE TABLE SILVER.DIM_SHOWS (
    -- Keys
    show_key NUMBER PRIMARY KEY DEFAULT SILVER.seq_show_key.NEXTVAL,
    imdb_id VARCHAR UNIQUE NOT NULL,
    
    -- Core Descriptors
    title VARCHAR,
    plot VARCHAR,
    
    -- AI-Enriched Fields
    plot_summary VARCHAR,
    plot_sentiment NUMBER(5,4),
    
    -- Awards Fields
    awards VARCHAR,
    total_wins NUMBER,
    total_nominations NUMBER,
    is_oscar_winner BOOLEAN,
    is_oscar_nominee BOOLEAN,
    is_emmy_winner BOOLEAN,
    is_emmy_nominee BOOLEAN,

    -- NEW: Denormalized Date Columns (Release)
    release_date DATE,
    release_year NUMBER,
    release_month NUMBER,
    release_day NUMBER,
    release_quarter NUMBER,
    release_day_of_week VARCHAR,

    -- NEW: Denormalized Date Columns (Added to Disney+)
    added_date DATE,
    added_year NUMBER,
    added_month NUMBER,
    added_day NUMBER,
    added_quarter NUMBER,
    added_day_of_week VARCHAR,
    
    -- Foreign Keys
    type_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_TYPES(type_key),
    rating_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_RATINGS(rating_key)
);

--==============================================================
-- 5. FACT TABLE
--==============================================================
CREATE OR REPLACE TABLE SILVER.FCT_SHOW_METRICS (
    show_key NUMBER PRIMARY KEY FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    year NUMBER(4),
    runtime_minutes NUMBER,
    metascore NUMBER,
    imdb_rating NUMBER(3,1),
    imdb_votes NUMBER
);

--==============================================================
-- 6. BRIDGE TABLES
--==============================================================
CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_GENRES (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    genre_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_GENRES(genre_key),
    PRIMARY KEY (show_key, genre_key)
);

CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_ACTORS (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    person_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_PERSONS(person_key),
    PRIMARY KEY (show_key, person_key)
);

CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_DIRECTORS (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    person_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_PERSONS(person_key),
    PRIMARY KEY (show_key, person_key)
);

CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_WRITERS (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    person_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_PERSONS(person_key),
    PRIMARY KEY (show_key, person_key)
);

CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_LANGUAGES (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    language_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_LANGUAGES(language_key),
    PRIMARY KEY (show_key, language_key)
);

CREATE OR REPLACE TABLE SILVER.BRIDGE_SHOW_COUNTRIES (
    show_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_SHOWS(show_key),
    country_key NUMBER FOREIGN KEY REFERENCES SILVER.DIM_COUNTRIES(country_key),
    PRIMARY KEY (show_key, country_key)
);

## DDL for streams to capture changes in Silver layer (data to be loaded into gold layer tables)

In [ ]:
CREATE OR REPLACE STREAM SILVER.STREAM_AUDIENCE_ENGAGEMENT
ON TABLE SILVER.FCT_SHOW_METRICS;


In [ ]:
CREATE OR REPLACE STREAM SILVER.STREAM_GENRE_TIMING
ON TABLE SILVER.FCT_SHOW_METRICS;


In [ ]:
CREATE OR REPLACE STREAM SILVER.STREAM_POWER_DUOS
ON TABLE SILVER.FCT_SHOW_METRICS;

## DDL for aggregated tables in Gold layer

In [ ]:
USE schema GOLD;

CREATE OR REPLACE TABLE GOLD.GENRE_RELEASE_TIMING_PERFORMANCE AS
SELECT
    g.genre_name,
    s.release_month,
    s.release_quarter,
    AVG(f.imdb_rating) AS avg_rating,
    SUM(f.imdb_votes) AS total_votes,
    COUNT(*) AS total_titles
FROM SILVER.DIM_SHOWS s
JOIN SILVER.FCT_SHOW_METRICS f
    ON s.show_key = f.show_key
JOIN SILVER.BRIDGE_SHOW_GENRES bg
    ON s.show_key = bg.show_key
JOIN SILVER.DIM_GENRES g
    ON bg.genre_key = g.genre_key
GROUP BY g.genre_name, s.release_month, s.release_quarter
ORDER BY g.genre_name, s.release_month;


In [ ]:

CREATE OR REPLACE TABLE GOLD.AUDIENCE_ENGAGEMENT_DECADE_SENTIMENT AS
SELECT
    (s.release_year - MOD(s.release_year, 10)) AS decade,
    g.genre_name,
    AVG(s.plot_sentiment) AS avg_sentiment,
    AVG(f.imdb_rating) AS avg_rating,
    SUM(f.imdb_votes) AS total_votes,
    COUNT(*) AS total_titles
FROM SILVER.DIM_SHOWS s
JOIN SILVER.FCT_SHOW_METRICS f
    ON s.show_key = f.show_key
JOIN SILVER.BRIDGE_SHOW_GENRES bg
    ON s.show_key = bg.show_key
JOIN SILVER.DIM_GENRES g
    ON bg.genre_key = g.genre_key
GROUP BY decade, g.genre_name
ORDER BY decade, g.genre_name;


In [ ]:
CREATE OR REPLACE TABLE GOLD.POWER_DUOS_PERFORMANCE AS
SELECT
    t.type_name,
    p_dir.person_name AS director_name,
    p_wr.person_name  AS writer_name,
    AVG(f.imdb_rating) AS avg_rating,
    SUM(f.imdb_votes) AS total_votes,
    COUNT(*) AS project_count
FROM SILVER.FCT_SHOW_METRICS f
JOIN SILVER.DIM_SHOWS s
    ON f.show_key = s.show_key
JOIN SILVER.DIM_TYPES t
    ON s.type_key = t.type_key

-- Director
JOIN SILVER.BRIDGE_SHOW_DIRECTORS bd
    ON s.show_key = bd.show_key
JOIN SILVER.DIM_PERSONS p_dir
    ON bd.person_key = p_dir.person_key

-- Writer
JOIN SILVER.BRIDGE_SHOW_WRITERS bw
    ON s.show_key = bw.show_key
JOIN SILVER.DIM_PERSONS p_wr
    ON bw.person_key = p_wr.person_key

GROUP BY t.type_name, p_dir.person_name, p_wr.person_name
ORDER BY avg_rating DESC;


### Stored procedure to load data from raw tables to enhanced table in bronze

In [ ]:
CREATE OR REPLACE PROCEDURE BRONZE.sp_process_bronze_data()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    result_message VARCHAR;
    raw_count INTEGER;
    enhanced_count INTEGER;
BEGIN
    SELECT COUNT(*) INTO enhanced_count FROM BRONZE.disney_plus_shows_enhanced;

    -- =============================================
    -- STEP 2: PROCESS INTO ENHANCED TABLE
    -- =============================================
    
    INSERT INTO BRONZE.disney_plus_shows_enhanced
    SELECT 
        imdb_id,
        title,
        plot,
        type,
        rated,
        
        -- Clean year
        CASE 
            WHEN year REGEXP '^[0-9]{4}' THEN REGEXP_SUBSTR(year, '^[0-9]{4}')::INTEGER
            ELSE NULL 
        END as year,
        
        -- Convert dates
        TRY_TO_DATE(released_at, 'DD Mon YYYY') as release_date,
        TRY_TO_DATE(added_at, 'MMMM DD, YYYY') as added_date,
        
        -- Extract runtime minutes
        TRY_CAST(REGEXP_SUBSTR(runtime, '^[0-9]+') AS INTEGER) as runtime_minutes,
        
        -- Original multi-value columns
        genre,
        director,
        writer,
        actors,
        language,
        country,
        awards,
        
        -- Awards parsing
        TRY_TO_NUMBER(REGEXP_SUBSTR(awards, '(\\d+) wins', 1, 1, 'e', 1)) as total_wins,
        TRY_TO_NUMBER(REGEXP_SUBSTR(awards, '(\\d+) nominations', 1, 1, 'e', 1)) as total_nominations,
        awards LIKE '%Won % Oscar%' as is_oscar_winner,
        awards LIKE '%Nominated for % Oscar%' as is_oscar_nominee,
        awards LIKE '%Won % Emmy%' as is_emmy_winner,
        awards LIKE '%Nominated for % Emmy%' as is_emmy_nominee,
        awards LIKE '%Won % Golden Globe%' as is_goldenglobe_winner,
        awards LIKE '%Nominated for % Golden Globe%' as is_goldenglobe_nominee,
        
        -- Cleaned numerical metrics
        CASE WHEN metascore REGEXP '^[0-9]+$' THEN metascore::INTEGER ELSE NULL END as metascore,
        CASE WHEN imdb_rating REGEXP '^[0-9]+\\.[0-9]+$' THEN imdb_rating::NUMBER(3,1) ELSE NULL END as imdb_rating,
        CASE WHEN imdb_votes REGEXP '^[0-9,]+$' THEN REPLACE(imdb_votes, ',', '')::INTEGER ELSE NULL END as imdb_votes,
        
        -- AI functions
        SNOWFLAKE.CORTEX.SENTIMENT(plot) as plot_sentiment,
        SNOWFLAKE.CORTEX.SUMMARIZE(plot) as plot_summary,
        
        -- Sentiment categorization
        CASE 
            WHEN plot_sentiment < -0.3 THEN 'Negative'
            WHEN plot_sentiment BETWEEN -0.3 AND 0.3 THEN 'Neutral'
            WHEN plot_sentiment > 0.3 THEN 'Positive'
            ELSE 'Unknown'
        END as sentiment_category
        
    FROM BRONZE.disney_plus_shows_raw
    WHERE imdb_id NOT IN (SELECT imdb_id FROM BRONZE.disney_plus_shows_enhanced); -- Incremental logic
    
    SELECT COUNT(*) INTO enhanced_count FROM BRONZE.disney_plus_shows_enhanced;
    result_message := result_message || 'Enhanced data: ' || enhanced_count || ' rows. ';
    
    RETURN result_message;

EXCEPTION
    WHEN OTHER THEN
        RETURN 'Error in bronze data processing: ' || SQLERRM;
END;
$$;

### Stored procedure to load data from enhanced table in bronze to tables in silver schema.

In [ ]:
CREATE OR REPLACE PROCEDURE SILVER.sp_process_enhanced_to_silver()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    res VARCHAR;
BEGIN
    -- 0. Check if Stream has data (Cost optimization)
    IF (NOT SYSTEM$STREAM_HAS_DATA('BRONZE.disney_plus_shows_enhanced_stream')) THEN
        RETURN 'Stream is empty. No new data to process.';
    END IF;

    -- =========================================================================
    -- 1. CAPTURE STREAM DATA INTO A TEMP TABLE (The Fix!)
    -- This consumes the stream ONCE and holds the data for the rest of the script.
    -- =========================================================================
    CREATE OR REPLACE TEMPORARY TABLE SILVER.temp_stream_buffer AS
    SELECT * FROM BRONZE.disney_plus_shows_enhanced_stream;

    -- From now on, we read from SILVER.temp_stream_buffer, NOT the stream directly.

    -- =========================================================================
    -- 2. LOAD LOOKUP DIMENSIONS
    -- =========================================================================
    
    INSERT INTO SILVER.DIM_TYPES (type_name) 
    SELECT DISTINCT type FROM SILVER.temp_stream_buffer 
    WHERE type IS NOT NULL AND type NOT IN (SELECT type_name FROM SILVER.DIM_TYPES);
    
    INSERT INTO SILVER.DIM_RATINGS (rating_name) 
    SELECT DISTINCT rated FROM SILVER.temp_stream_buffer 
    WHERE rated IS NOT NULL AND rated NOT IN (SELECT rating_name FROM SILVER.DIM_RATINGS);
    
    INSERT INTO SILVER.DIM_GENRES (genre_name) 
    SELECT DISTINCT TRIM(VALUE) FROM SILVER.temp_stream_buffer, 
    LATERAL FLATTEN(input=>SPLIT(genre, ',')) 
    WHERE TRIM(VALUE) != '' AND TRIM(VALUE) NOT IN (SELECT genre_name FROM SILVER.DIM_GENRES);

    INSERT INTO SILVER.DIM_LANGUAGES (language_name) 
    SELECT DISTINCT TRIM(VALUE) FROM SILVER.temp_stream_buffer, 
    LATERAL FLATTEN(input=>SPLIT(language, ',')) 
    WHERE TRIM(VALUE) != '' AND TRIM(VALUE) NOT IN (SELECT language_name FROM SILVER.DIM_LANGUAGES);

    INSERT INTO SILVER.DIM_COUNTRIES (country_name) 
    SELECT DISTINCT TRIM(VALUE) FROM SILVER.temp_stream_buffer, 
    LATERAL FLATTEN(input=>SPLIT(country, ',')) 
    WHERE TRIM(VALUE) != '' AND TRIM(VALUE) NOT IN (SELECT country_name FROM SILVER.DIM_COUNTRIES);

    INSERT INTO SILVER.DIM_PERSONS (person_name) 
    SELECT DISTINCT person_name FROM (
        SELECT TRIM(VALUE) as person_name FROM SILVER.temp_stream_buffer, LATERAL FLATTEN(input=>SPLIT(actors, ','))
        UNION
        SELECT TRIM(VALUE) FROM SILVER.temp_stream_buffer, LATERAL FLATTEN(input=>SPLIT(director, ','))
        UNION
        SELECT TRIM(VALUE) FROM SILVER.temp_stream_buffer, LATERAL FLATTEN(input=>SPLIT(writer, ','))
    ) WHERE person_name != '' AND person_name != 'N/A' AND person_name NOT IN (SELECT person_name FROM SILVER.DIM_PERSONS);

    -- =========================================================================
    -- 3. LOAD CORE DIMENSION (DIM_SHOWS)
    -- =========================================================================
    MERGE INTO SILVER.DIM_SHOWS t
    USING (
        SELECT 
            s.imdb_id, s.title, s.plot, s.awards,
            s.plot_summary, s.plot_sentiment,
            s.total_wins, s.total_nominations, 
            s.is_oscar_winner, s.is_oscar_nominee, 
            s.is_emmy_winner, s.is_emmy_nominee,
            
            dt.type_key, 
            dr.rating_key,
            
            s.release_date,
            YEAR(s.release_date) as release_year,
            MONTH(s.release_date) as release_month,
            DAY(s.release_date) as release_day,
            QUARTER(s.release_date) as release_quarter,
            DAYNAME(s.release_date) as release_day_of_week,

            s.added_date,
            YEAR(s.added_date) as added_year,
            MONTH(s.added_date) as added_month,
            DAY(s.added_date) as added_day,
            QUARTER(s.added_date) as added_quarter,
            DAYNAME(s.added_date) as added_day_of_week

        FROM SILVER.temp_stream_buffer s -- Reading from Temp Table
        LEFT JOIN SILVER.DIM_TYPES dt ON s.type = dt.type_name
        LEFT JOIN SILVER.DIM_RATINGS dr ON s.rated = dr.rating_name
    ) source_data 
    ON t.imdb_id = source_data.imdb_id
    WHEN NOT MATCHED THEN INSERT (
        show_key, imdb_id, title, plot, awards, 
        plot_summary, plot_sentiment,
        total_wins, total_nominations, 
        is_oscar_winner, is_oscar_nominee, is_emmy_winner, is_emmy_nominee,
        type_key, rating_key,
        
        release_date, release_year, release_month, release_day, release_quarter, release_day_of_week,
        added_date, added_year, added_month, added_day, added_quarter, added_day_of_week
    ) VALUES (
        SILVER.seq_show_key.NEXTVAL, 
        source_data.imdb_id, source_data.title, source_data.plot, source_data.awards, 
        source_data.plot_summary, source_data.plot_sentiment,
        source_data.total_wins, source_data.total_nominations, 
        source_data.is_oscar_winner, source_data.is_oscar_nominee, source_data.is_emmy_winner, source_data.is_emmy_nominee,
        source_data.type_key, source_data.rating_key,
        
        source_data.release_date, source_data.release_year, source_data.release_month, source_data.release_day, source_data.release_quarter, source_data.release_day_of_week,
        source_data.added_date, source_data.added_year, source_data.added_month, source_data.added_day, source_data.added_quarter, source_data.added_day_of_week
    );

    -- =========================================================================
    -- 4. LOAD FACT TABLE (FCT_SHOW_METRICS)
    -- =========================================================================
    MERGE INTO SILVER.FCT_SHOW_METRICS t
    USING (
        SELECT ds.show_key, e.year, e.runtime_minutes, e.metascore, e.imdb_rating, e.imdb_votes
        FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
        JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id
    ) s ON t.show_key = s.show_key
    WHEN NOT MATCHED THEN INSERT (show_key, year, runtime_minutes, metascore, imdb_rating, imdb_votes)
    VALUES (s.show_key, s.year, s.runtime_minutes, s.metascore, s.imdb_rating, s.imdb_votes);

    -- =========================================================================
    -- 5. LOAD BRIDGE TABLES
    -- =========================================================================
    
    INSERT INTO SILVER.BRIDGE_SHOW_GENRES (show_key, genre_key)
    SELECT DISTINCT ds.show_key, dg.genre_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.genre, ',')) g
    JOIN SILVER.DIM_GENRES dg ON dg.genre_name = TRIM(g.VALUE)
    WHERE (ds.show_key, dg.genre_key) NOT IN (SELECT show_key, genre_key FROM SILVER.BRIDGE_SHOW_GENRES);

    INSERT INTO SILVER.BRIDGE_SHOW_ACTORS (show_key, person_key)
    SELECT DISTINCT ds.show_key, dp.person_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.actors, ',')) a
    JOIN SILVER.DIM_PERSONS dp ON dp.person_name = TRIM(a.VALUE)
    WHERE (ds.show_key, dp.person_key) NOT IN (SELECT show_key, person_key FROM SILVER.BRIDGE_SHOW_ACTORS);

    INSERT INTO SILVER.BRIDGE_SHOW_DIRECTORS (show_key, person_key)
    SELECT DISTINCT ds.show_key, dp.person_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.director, ',')) d
    JOIN SILVER.DIM_PERSONS dp ON dp.person_name = TRIM(d.VALUE)
    WHERE (ds.show_key, dp.person_key) NOT IN (SELECT show_key, person_key FROM SILVER.BRIDGE_SHOW_DIRECTORS);

    INSERT INTO SILVER.BRIDGE_SHOW_WRITERS (show_key, person_key)
    SELECT DISTINCT ds.show_key, dp.person_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.writer, ',')) w
    JOIN SILVER.DIM_PERSONS dp ON dp.person_name = TRIM(w.VALUE)
    WHERE (ds.show_key, dp.person_key) NOT IN (SELECT show_key, person_key FROM SILVER.BRIDGE_SHOW_WRITERS);

    INSERT INTO SILVER.BRIDGE_SHOW_LANGUAGES (show_key, language_key)
    SELECT DISTINCT ds.show_key, dl.language_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.language, ',')) l
    JOIN SILVER.DIM_LANGUAGES dl ON dl.language_name = TRIM(l.VALUE)
    WHERE (ds.show_key, dl.language_key) NOT IN (SELECT show_key, language_key FROM SILVER.BRIDGE_SHOW_LANGUAGES);

    INSERT INTO SILVER.BRIDGE_SHOW_COUNTRIES (show_key, country_key)
    SELECT DISTINCT ds.show_key, dc.country_key
    FROM SILVER.temp_stream_buffer e -- Reading from Temp Table
    JOIN SILVER.DIM_SHOWS ds ON e.imdb_id = ds.imdb_id,
    LATERAL FLATTEN(input=>SPLIT(e.country, ',')) c
    JOIN SILVER.DIM_COUNTRIES dc ON dc.country_name = TRIM(c.VALUE)
    WHERE (ds.show_key, dc.country_key) NOT IN (SELECT show_key, country_key FROM SILVER.BRIDGE_SHOW_COUNTRIES);

    res := 'Silver Layer processing complete.';
    RETURN res;
END;
$$;

### Stored procedures to load data into gold tables from silver (one sp per use case)

In [ ]:
CREATE OR REPLACE PROCEDURE GOLD.SP_LOAD_GENRE_RELEASE_TIMING()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    INSERT INTO GOLD.GENRE_RELEASE_TIMING_PERFORMANCE
    SELECT
        g.genre_name,
        s.release_month,
        s.release_quarter,
        AVG(f.imdb_rating),
        SUM(f.imdb_votes),
        COUNT(*)
    FROM SILVER.STREAM_GENRE_TIMING sm
    JOIN SILVER.FCT_SHOW_METRICS f       ON sm.show_key = f.show_key
    JOIN SILVER.DIM_SHOWS s              ON sm.show_key = s.show_key
    JOIN SILVER.BRIDGE_SHOW_GENRES bg    ON s.show_key = bg.show_key
    JOIN SILVER.DIM_GENRES g             ON bg.genre_key = g.genre_key
    GROUP BY g.genre_name, s.release_month, s.release_quarter;

    RETURN 'Genre release timing incremental load complete.';
END;
$$;


In [ ]:
CREATE OR REPLACE PROCEDURE GOLD.SP_LOAD_AUDIENCE_ENGAGEMENT()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    INSERT INTO GOLD.AUDIENCE_ENGAGEMENT_DECADE_SENTIMENT
    SELECT
        (s.release_year - MOD(s.release_year, 10)) AS decade,
        g.genre_name,
        AVG(s.plot_sentiment),
        AVG(f.imdb_rating),
        SUM(f.imdb_votes),
        COUNT(*)
    FROM SILVER.STREAM_AUDIENCE_ENGAGEMENT sm
    JOIN SILVER.FCT_SHOW_METRICS f   ON sm.show_key = f.show_key
    JOIN SILVER.DIM_SHOWS s          ON sm.show_key = s.show_key
    JOIN SILVER.BRIDGE_SHOW_GENRES bg ON s.show_key = bg.show_key
    JOIN SILVER.DIM_GENRES g         ON bg.genre_key = g.genre_key
    GROUP BY decade, g.genre_name;

    RETURN 'Audience engagement incremental load complete.';
END;
$$;


In [ ]:
CREATE OR REPLACE PROCEDURE GOLD.SP_LOAD_POWER_DUOS()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    INSERT INTO GOLD.POWER_DUOS_PERFORMANCE
    SELECT
        t.type_name,
        p_dir.person_name AS director_name,
        p_wr.person_name  AS writer_name,
        AVG(f.imdb_rating),
        SUM(f.imdb_votes),
        COUNT(*)
    FROM SILVER.STREAM_POWER_DUOS sm
    JOIN SILVER.FCT_SHOW_METRICS f            ON sm.show_key = f.show_key
    JOIN SILVER.DIM_SHOWS s                   ON sm.show_key = s.show_key
    JOIN SILVER.DIM_TYPES t                   ON s.type_key = t.type_key
    
    -- Director
    JOIN SILVER.BRIDGE_SHOW_DIRECTORS bd      ON s.show_key = bd.show_key
    JOIN SILVER.DIM_PERSONS p_dir             ON bd.person_key = p_dir.person_key

    -- Writer
    JOIN SILVER.BRIDGE_SHOW_WRITERS bw        ON s.show_key = bw.show_key
    JOIN SILVER.DIM_PERSONS p_wr              ON bw.person_key = p_wr.person_key

    GROUP BY t.type_name, p_dir.person_name, p_wr.person_name;

    RETURN 'Director–Writer duos incremental load complete.';
END;
$$;


### Truncate commands for tables in bronze and silver in case we want to do a intial load again (for the streams to track data changes we need to load from the start)

In [ ]:

-- -- 1. EMPTY Enhanced Btonze  TABLES
-- TRUNCATE table BRONZE.disney_plus_shows_enhanced
-- -- ==========================================
-- -- 2. EMPTY ALL SILVER TABLES
-- -- ==========================================

-- -- A. Clear the Fact Table first (Child of DIM_SHOWS)
-- TRUNCATE TABLE SILVER.FCT_SHOW_METRICS;

-- -- B. Clear all Bridge Tables (Children of DIM_SHOWS & Lookups)
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_GENRES;
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_ACTORS;
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_DIRECTORS;
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_WRITERS;
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_LANGUAGES;
-- TRUNCATE TABLE SILVER.BRIDGE_SHOW_COUNTRIES;

-- -- C. Clear the Main Dimension (Parent of Bridges/Fact)
-- TRUNCATE TABLE SILVER.DIM_SHOWS;

-- -- D. Clear the Lookup Dimensions
-- TRUNCATE TABLE SILVER.DIM_GENRES;
-- TRUNCATE TABLE SILVER.DIM_PERSONS;
-- TRUNCATE TABLE SILVER.DIM_LANGUAGES;
-- TRUNCATE TABLE SILVER.DIM_COUNTRIES;
-- TRUNCATE TABLE SILVER.DIM_TYPES;
-- TRUNCATE TABLE SILVER.DIM_RATINGS;

-- -- ==========================================
-- -- 2. RESET THE SEQUENCE
-- -- ==========================================
-- -- This ensures your show_key starts counting from 1 again
-- CREATE OR REPLACE SEQUENCE SILVER.seq_show_key START = 1 INCREMENT = 1;

## Load data into Raw table in Bronze from CSV in stage.

In [ ]:
-- Initial load
COPY INTO BRONZE.disney_plus_shows_raw
FROM @BRONZE.BRONZE/disney_plus_shows.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

In [ ]:
--Incremental load (this part is done at the end after whole intial load is done till gold layer)

COPY INTO BRONZE.disney_plus_shows_raw
FROM @BRONZE.BRONZE/New_unique_movies.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

In [ ]:
select count(*) from BRONZE.disney_plus_shows_raw;

In [ ]:
select title,plot_sentiment,plot_summary,sentiment_category from BRONZE.disney_plus_shows_enhanced limit 5;

### Stored Procedure calls and data validation (Run all of them in order to load data right from raw table to all tables in gold layer for initial or incremental load)

In [ ]:

-- Run data processing (works for initial AND incremental)
CALL BRONZE.sp_process_bronze_data();


In [ ]:
-- Verify results
SELECT * FROM BRONZE.disney_plus_shows_enhanced LIMIT 5;

In [ ]:
describe table BRONZE.disney_plus_shows_enhanced

In [ ]:
SELECT SYSTEM$STREAM_HAS_DATA('BRONZE.disney_plus_shows_enhanced_stream');

In [ ]:
-- Stored proc to load data from bronze to silver
CALL SILVER.sp_process_enhanced_to_silver();

In [ ]:
select count(*) from silver.dim_shows

In [ ]:
select count(*) from silver.fct_show_metrics

In [ ]:
select * from silver.dim_shows limit 5;

In [ ]:
-- stored proc call to load data from silver into gold tables
call GOLD.SP_LOAD_POWER_DUOS()

In [ ]:
call GOLD.SP_LOAD_GENRE_RELEASE_TIMING()

In [ ]:
call GOLD.SP_LOAD_AUDIENCE_ENGAGEMENT()

In [ ]:
SELECT count(*) FROM GOLD.AUDIENCE_ENGAGEMENT_DECADE_SENTIMENT

In [ ]:
SELECT * FROM GOLD.AUDIENCE_ENGAGEMENT_DECADE_SENTIMENT order by decade nulls last LIMIT 5;

In [ ]:
select count(*) from GOLD.GENRE_RELEASE_TIMING_PERFORMANCE;

In [ ]:
select * from GOLD.GENRE_RELEASE_TIMING_PERFORMANCE limit 10;

In [ ]:
select count(*) from GOLD.POWER_DUOS_PERFORMANCE;

In [ ]:
select * from GOLD.POWER_DUOS_PERFORMANCE order by avg_rating desc nulls last limit 5;

## CORTEX SEARCH IN BRONZE LAYER

In [ ]:
GRANT USAGE ON CORTEX SEARCH SERVICE BRONZE.DISNEY_SEARCH_ANALYST TO ROLE ROLE_TEAM_BSJ;

In [ ]:
SHOW CORTEX SEARCH SERVICES IN SCHEMA BRONZE;

In [ ]:
-- Check service metadata to see what columns were included
SELECT *
FROM TABLE(
  CORTEX_SEARCH_DATA_SCAN(
    SERVICE_NAME => 'DB_TEAM_BSJ.BRONZE.disney_plot_search'
  )
)
LIMIT 1;

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'DB_TEAM_BSJ.BRONZE.disney_plot_search',
    '{
      "query": "teenage romance",
      "columns": ["title","plot", "genre", "sentiment_category", "imdb_rating"],
      "limit": 5
    }'
  )
)['results'] as results;

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'DB_TEAM_BSJ.BRONZE.disney_plot_search',
    '{
      "query": "animated movie", 
      "columns": ["title","plot", "genre", "sentiment_category", "imdb_rating"],
      "filter": {"@eq": {"genre": "Animation"}},
      "limit": 5
    }'
  )
)['results'] as results;

In [ ]:
-- Cortex Search understands concepts
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'DB_TEAM_BSJ.BRONZE.disney_plot_search',
    '{
      "query": "family holiday stories",
      "columns": ["title","plot", "genre", "sentiment_category", "imdb_rating"],
      "limit": 5
    }'
  )
)['results'] as results;


In [ ]:
-- Compare with traditional SQL (will miss many relevant results)
SELECT title, plot, genre, release_date
FROM BRONZE.disney_plus_shows_enhanced
WHERE plot ILIKE '%family%' AND plot ILIKE '%holiday%'
LIMIT 5;

### Cortex Analyst (Disney_Magic_Mirror) created from UI and more than 10 queries asked and have been added to verified queries. 

### Views in gold layer to be used for streamlit dashboards (created 3 different interactive streamlit apps for 3 use cases)

In [ ]:
CREATE OR REPLACE VIEW GOLD.RANKED_MONTHS_PER_GENRE AS
WITH ranked_data AS (
    SELECT
        genre_name,
        release_month,
        avg_rating,
        total_votes,
        total_titles,
        -- Rank months for EACH genre.
        -- Criteria: Highest Rating first, then Most Votes (as tie-breaker)
        ROW_NUMBER() OVER (
            PARTITION BY genre_name
            ORDER BY avg_rating DESC, total_votes DESC
        ) AS rank
    FROM GOLD.GENRE_RELEASE_TIMING_PERFORMANCE
)
SELECT
    genre_name,
    release_month,
    avg_rating,
    total_votes,
    total_titles,
    rank
FROM ranked_data WHERE rank <= 3;

In [ ]:
select * from gold.RANKED_MONTHS_PER_GENRE limit 5;

In [ ]:
CREATE OR REPLACE VIEW GOLD.top3_genres_by_decade AS
WITH decade_genre_stats AS (
    SELECT 
        CASE 
            WHEN f.year BETWEEN 1980 AND 1989 THEN '1980s'
            WHEN f.year BETWEEN 1990 AND 1999 THEN '1990s'
            WHEN f.year BETWEEN 2000 AND 2009 THEN '2000s'
            WHEN f.year BETWEEN 2010 AND 2019 THEN '2010s'
            WHEN f.year >= 2020 THEN '2020s'
            ELSE 'Other Years'
        END as decade,
        g.genre_name,
        COUNT(*) as title_count,
        ROUND(AVG(f.imdb_rating), 2) as avg_rating,
        SUM(f.imdb_votes) as total_votes
    FROM SILVER.DIM_SHOWS ds
    JOIN SILVER.FCT_SHOW_METRICS f ON ds.show_key = f.show_key
    JOIN SILVER.BRIDGE_SHOW_GENRES bg ON ds.show_key = bg.show_key
    JOIN SILVER.DIM_GENRES g ON bg.genre_key = g.genre_key
    WHERE f.year IS NOT NULL AND f.year::STRING REGEXP '^[0-9]+$'
    GROUP BY decade, g.genre_name
    HAVING title_count >= 3
)
SELECT 
    decade,
    genre_name,
    title_count,
    avg_rating,
    total_votes,
    RANK() OVER (PARTITION BY decade ORDER BY avg_rating DESC) as rating_rank
FROM decade_genre_stats
QUALIFY rating_rank <= 3
ORDER BY 
    CASE 
        WHEN decade = 'Other Years' THEN 999
        ELSE CAST(SUBSTR(decade, 1, 4) AS INTEGER)
    END,
    rating_rank;

In [ ]:
SELECT * FROM GOLD.top3_genres_by_decade limit 5;

In [ ]:
CREATE OR REPLACE VIEW GOLD.V_DIRECTOR_WRITER_IMPACT AS
WITH Director_Baseline AS (
    SELECT
        director_name,
        type_name,
        SUM(project_count) as total_projects_directed,
        ROUND(SUM(avg_rating * project_count) / SUM(project_count), 2) as director_avg_rating
    FROM GOLD.POWER_DUOS_PERFORMANCE
    GROUP BY director_name, type_name
),
Duo_Comparison AS (
    SELECT
        t.type_name,
        t.director_name,
        t.writer_name,
        t.project_count as duo_projects,
        ROUND(t.avg_rating, 2) as duo_rating
    FROM GOLD.POWER_DUOS_PERFORMANCE t
    -- FILTER REMOVED: Now showing ALL collaborations, even single ones
)
SELECT
    d.director_name,
    d.type_name,
    d.director_avg_rating AS director_baseline,
    duo.writer_name,
    duo.duo_projects,
    duo.duo_rating,
    ROUND(duo.duo_rating - d.director_avg_rating, 2) AS rating_impact_score
FROM Duo_Comparison duo
JOIN Director_Baseline d 
    ON duo.director_name = d.director_name 
    AND duo.type_name = d.type_name;

In [ ]:
select * from GOLD.V_DIRECTOR_WRITER_IMPACT order by rating_impact_score desc nulls last limit 10 ;